# 03. Признаки, baseline и валидация

Ноутбук собирает признаки без утечки будущих продаж и сравнивает несколько простых baseline-моделей по `net_sales_qty`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.baselines import (
    median_by_weekday,
    moving_average_7,
    moving_average_28,
    naive_last_value,
    seasonal_naive_7,
    seasonal_naive_28,
)
from src.config import PROCESSED_DATA_DIR, RESULTS_DIR
from src.features import build_feature_matrix
from src.metrics import metrics_table

In [2]:
mart_daily_sales = pd.read_csv(PROCESSED_DATA_DIR / 'mart_daily_sales.csv', parse_dates=['sales_date'])
features = build_feature_matrix(mart_daily_sales)
features.to_csv(PROCESSED_DATA_DIR / 'features_lags_rolling.csv', index=False)
features.head()

,sales_date,stock_code,description,market_id,sales_qty,avg_unit_price,revenue,invoices_cnt,customers_cnt,returns_qty,...,customers_cnt_lag_7,rolling_mean_7,rolling_mean_28,rolling_std_28,rolling_median_28,avg_unit_price_lag_7,price_change_abs,price_change_pct,days_since_last_sale,zero_sales_streak
0,2010-03-04,10002,INFLATABLE POLITICAL GLOBE,Australia,12,0.85,10.2,1,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,2010-10-26,10002,INFLATABLE POLITICAL GLOBE,Australia,24,0.85,20.4,1,1,0,...,NaN,12.0,12.0,NaN,12.0,NaN,NaN,NaN,236.0,0
2,2010-01-11,10002,INFLATABLE POLITICAL GLOBE,Denmark,48,0.85,40.8,1,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,2010-04-22,10002,INFLATABLE POLITICAL GLOBE,Denmark,200,0.72,144.0,1,1,0,...,NaN,48.0,48.0,NaN,48.0,NaN,NaN,NaN,101.0,0
4,2010-11-02,10002,INFLATABLE POLITICAL GLOBE,Denmark,24,0.85,20.4,1,1,0,...,NaN,124.0,124.0,107.480231,124.0,NaN,NaN,NaN,194.0,0


## Baseline-модели

Все прогнозы используют только прошлые значения внутри пары `stock_code × market_id`.

In [4]:
baseline_frame = features[
    ['sales_date', 'stock_code', 'market_id', 'net_sales_qty']
].copy()

baseline_frame['sales_date'] = pd.to_datetime(baseline_frame['sales_date'])

baseline_frame = baseline_frame.sort_values(
    ['stock_code', 'market_id', 'sales_date'],
    kind='mergesort'
)

group_cols = ['stock_code', 'market_id']
group = baseline_frame.groupby(group_cols, sort=False)['net_sales_qty']

# Last value / seasonal naive
baseline_frame['naive_last_value'] = group.shift(1)
baseline_frame['seasonal_naive_7'] = group.shift(7)
baseline_frame['seasonal_naive_28'] = group.shift(28)

# Moving averages без leakage: сначала shift(1), потом rolling
shifted_target = group.shift(1)

baseline_frame['moving_average_7'] = (
    shifted_target
    .groupby([baseline_frame['stock_code'], baseline_frame['market_id']])
    .rolling(window=7, min_periods=1)
    .mean()
    .reset_index(level=[0, 1], drop=True)
)

baseline_frame['moving_average_28'] = (
    shifted_target
    .groupby([baseline_frame['stock_code'], baseline_frame['market_id']])
    .rolling(window=28, min_periods=1)
    .mean()
    .reset_index(level=[0, 1], drop=True)
)

# Median by weekday: медиана прошлых значений для той же группы и дня недели
baseline_frame['weekday'] = baseline_frame['sales_date'].dt.dayofweek

baseline_frame['median_by_weekday'] = (
    baseline_frame
    .groupby(['stock_code', 'market_id', 'weekday'], sort=False)['net_sales_qty']
    .transform(lambda s: s.shift(1).expanding(min_periods=1).median())
)

baseline_frame.head(10)

,sales_date,stock_code,market_id,net_sales_qty,naive_last_value,seasonal_naive_7,seasonal_naive_28,moving_average_7,moving_average_28,weekday,median_by_weekday
0,2010-03-04,10002,Australia,12,NaN,NaN,NaN,NaN,NaN,3,NaN
1,2010-10-26,10002,Australia,24,12.0,NaN,NaN,12.0,12.0,1,NaN
2,2010-01-11,10002,Denmark,48,NaN,NaN,NaN,NaN,NaN,0,NaN
3,2010-04-22,10002,Denmark,200,48.0,NaN,NaN,48.0,48.0,3,NaN
4,2010-11-02,10002,Denmark,24,200.0,NaN,NaN,124.0,124.0,1,NaN
5,2010-02-17,10002,EIRE,12,NaN,NaN,NaN,NaN,NaN,2,NaN
6,2010-06-03,10002,EIRE,12,12.0,NaN,NaN,12.0,12.0,3,NaN
7,2010-09-20,10002,EIRE,12,12.0,NaN,NaN,12.0,12.0,0,NaN
8,2010-10-14,10002,EIRE,12,12.0,NaN,NaN,12.0,12.0,3,12.0
9,2010-12-10,10002,EIRE,12,12.0,NaN,NaN,12.0,12.0,4,NaN


## Расчет метрик

In [5]:
metric_rows = []
baseline_columns = [
    'naive_last_value',
    'seasonal_naive_7',
    'seasonal_naive_28',
    'moving_average_7',
    'moving_average_28',
    'median_by_weekday',
]

for baseline_name in baseline_columns:
    valid = baseline_frame.dropna(subset=['net_sales_qty', baseline_name])
    table = metrics_table(valid['net_sales_qty'], valid[baseline_name])
    table['baseline'] = baseline_name
    metric_rows.append(table)

baseline_metrics = pd.concat(metric_rows, ignore_index=True)
baseline_metrics = baseline_metrics[['baseline', 'metric', 'value']]

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
baseline_metrics.to_csv(RESULTS_DIR / 'baseline_metrics.csv', index=False)
baseline_metrics

,baseline,metric,value
0,naive_last_value,wmape,1.157950
1,naive_last_value,mae,22.741378
2,naive_last_value,rmse,143.923766
3,naive_last_value,bias,0.022449
4,naive_last_value,service_level_proxy,0.588927
5,naive_last_value,stockout_risk_rate,0.411073
6,seasonal_naive_7,wmape,1.201674
7,seasonal_naive_7,mae,22.928303
8,seasonal_naive_7,rmse,131.232753
9,seasonal_naive_7,bias,0.020621


## Выводы

- Лучший baseline по WMAPE: `[A]`.
- Если forecast bias положительный, прогноз систематически завышает спрос; если отрицательный, занижает.
- Baseline нужен как честная точка сравнения: более сложный подход имеет смысл только если он устойчиво лучше простых правил.